# 🛰️ NN_segmentation — Kaggle Launcher

---

## 📋 Setup iniziale

Prima di eseguire, collega il dataset dal pannello **"+ Add Data"** (barra laterale destra):

| Dataset Kaggle da allegare | Come trovarlo |
|---|---|
| **Qualsiasi dataset in formato images/label** | Cerca: `global-land-cover-mapping-openearthmap` (o altri) |

> ✅ **Non devi caricare niente dal tuo PC!**  
> Il dataset OpenEarthMap è già disponibile pubblicamente su Kaggle.  


## ① Clone / Pull del codice da GitHub

In [ ]:
import os

REPO_URL  = 'https://github.com/robertopassante/NN_segmentation.git'
REPO_NAME = 'NN_segmentation'
REPO_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print(f'[INFO] Clonazione {REPO_NAME}...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'[INFO] Repo già presente. Aggiornamento...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f'[OK] Working directory: {os.getcwd()}')

## ② Installazione dipendenze e pesi pre-addestrati RSP Swin-T

In [ ]:
import os

!pip install -q -r requirements.txt
!pip install -q yacs safetensors rasterio albumentations

# Pesi RSP Swin-T (download solo se assenti)
WEIGHTS_PATH = '/kaggle/working/NN_segmentation/rsp-swin-t-ckpt.safetensors'
if not os.path.exists(WEIGHTS_PATH):
    print('[INFO] Download pesi RSP Swin-T...')
    !wget -q -O {WEIGHTS_PATH} \
        https://huggingface.co/BiliSakura/RSP-Swin-T/resolve/main/model.safetensors
    print('[OK] Pesi scaricati.')
else:
    print('[SKIP] Pesi RSP già presenti.')

## ③ Verifica dataset allegato

> Il sistema rileva automaticamente qualsiasi dataset strutturato in `images/train`.


In [ ]:
import os
import glob

# ── Rilevamento automatico della cartella del dataset ────────────────────────
# Troviamo automaticamente il path cercando la cartella 'images/train'
found_paths = glob.glob('/kaggle/input/**/images/train', recursive=True)
if found_paths:
    # Prende la cartella "padre" di images/train
    INPUT_DIR = os.path.dirname(os.path.dirname(found_paths[0]))
else:
    INPUT_DIR = '/kaggle/input/global-land-cover-mapping-openearthmap'

# Verifica che il dataset sia allegato
if not os.path.exists(INPUT_DIR) or not os.path.exists(os.path.join(INPUT_DIR, 'images')):
    print(f'❌ Dataset non trovato (path esplorato: {INPUT_DIR})')
    print(f'   → Vai su "+ Add Data" e cerca "global-land-cover-mapping-openearthmap"')
    raise FileNotFoundError(f'Dataset mancante: {INPUT_DIR}')

# Conta le immagini disponibili
for split in ['train', 'val']:
    img_dir = os.path.join(INPUT_DIR, 'images', split)
    lbl_dir = os.path.join(INPUT_DIR, 'label',  split)
    n_img = len([f for f in os.listdir(img_dir) if f.endswith('.tif')]) if os.path.exists(img_dir) else 0
    n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith('.tif')]) if os.path.exists(lbl_dir) else 0
    print(f'[{split}] Immagini: {n_img} | Mask: {n_lbl}')

print(f'\n✅ Dataset intercettato correttamente in: {INPUT_DIR}')

## ④ Training (Full Dataset)
L'addestramento userà tutte le immagini rilevate.

In [ ]:
%cd /kaggle/working/NN_segmentation
!python main_kaggle.py

## ⑤ Esporta output nel tab "Output" di Kaggle


In [ ]:
import os, shutil

REPO_DIR = '/kaggle/working/NN_segmentation'
OUT_DIR  = '/kaggle/working'

# Checkpoint e loss curve
for fname in ['best_model.pth', 'loss_curve.png']:
    src = os.path.join(REPO_DIR, fname)
    dst = os.path.join(OUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'[OK]  {fname} → {dst}')
    else:
        print(f'[SKIP] {fname} non trovato')

# Immagini di predizione per classe
pred_src = os.path.join(REPO_DIR, 'output_samples')
pred_dst = os.path.join(OUT_DIR,  'output_samples')
if os.path.exists(pred_src):
    shutil.copytree(pred_src, pred_dst, dirs_exist_ok=True)
    n = len(os.listdir(pred_dst))
    print(f'[OK]  output_samples/ ({n} immagini) → {pred_dst}')

print('\n✅ Output pronti nel tab "Output" di Kaggle.')

## ⑥ Fase 5: Generazione Pseudo-Labels (Advanced Task)
Per sbloccare il potenziale del modello (80%+ mIoU), useremo il modello appena addestrato in combinazione con SAM di Meta per auto-etichettare enormi quantità di nuove immagini.\n
**Nota:** Cambia `UNLABELED_DIR` con il path di una cartella contenente immagini senza maschere.

In [ ]:
import os
%cd /kaggle/working/NN_segmentation

# Scegli la cartella delle immagini senza etichetta da processare
UNLABELED_DIR = '/kaggle/input/global-land-cover-mapping-openearthmap/images/train'  # Da cambiare con il tuo nuovo dataset unlabeled!
OUT_PSEUDO_DIR = '/kaggle/working/pseudo_dataset'

# 1. Download pesi SAM se non presenti
SAM_WEIGHTS = '/kaggle/working/NN_segmentation/sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_WEIGHTS):
    print('[INFO] Scarico Segment Anything Model (SAM) weights...')
    !wget -q -O {SAM_WEIGHTS} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

# 2. Esegui il processo massivo
!python pseudo_labeling.py \
    --input_dir {UNLABELED_DIR} \
    --output_dir {OUT_PSEUDO_DIR} \
    --unet_ckpt /kaggle/working/NN_segmentation/best_model.pth \
    --sam_ckpt {SAM_WEIGHTS}

print('\n✅ Pseudo-Labels generate con successo nella cartella:', OUT_PSEUDO_DIR)